# Proposed Citi Bike task and ingestion

**Contributors:** initial scaffold and local verification prepared with AI assistance. Team reviewers and substantive changes will be recorded after review. See [the assistance record](../docs/AI_USE.md).

**Proposal; not executed.** No archives have been downloaded. Confirm the scope with the team.

Proposed prediction: next-day citywide NYC ride count, using calendar variables and counts available through the previous day. Use complete NYC months and keep Jersey City outside this scope.

Initial hypothesis: weekly demand patterns are repeatable enough to support forecasting. Investigate the pattern with an appropriate statistical procedure before committing to the task. A graph alone is not evidence, and a negative result is allowed.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'src' / 'mushrooms.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the repository folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from citibike import download_and_aggregate, one_step_features
# Copy official NYC ZIP URLs linked from https://citibikenyc.com/system-data.
ARCHIVE_URLS = []
if not ARCHIVE_URLS:
    raise RuntimeError('Choose complete official NYC months with the group, then fill ARCHIVE_URLS.')

## Programmatic download and aggregation

The loader streams all direct CSV members in each archive and reads in chunks. It validates timestamp parsing and refuses silent zero-filling. Audit archive coverage, duplicate ride IDs, source versions and timezone treatment before treating this as final preparation.

In [ ]:
import pandas as pd
daily_parts = [download_and_aggregate(url) for url in ARCHIVE_URLS]
daily = pd.concat(daily_parts).sort_index()
if daily.index.duplicated().any():
    raise ValueError('Date ranges overlap. Resolve duplicate coverage before merging.')
expected = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
if not daily.index.equals(expected.rename('date')):
    raise ValueError('Selected months do not form a complete contiguous date range.')
(ROOT/'data/processed').mkdir(parents=True, exist_ok=True)
daily.to_csv(ROOT/'data/processed/citibike_daily.csv')
display(daily.head())

## Features and evaluation design

Every lag points backwards. Shifting before a rolling mean prevents using the target day in its own prediction. Use chronological train/validation/test blocks and rolling-origin validation. Compare yesterday and same-weekday-last-week baselines against tuned regression/forecasting models using MAE and RMSE. One-step evaluation can use each newly observed actual day; a multi-day forecast from a fixed origin cannot use future actual lags. Document which setting you implement.

In [ ]:
features = one_step_features(daily)
first = int(len(features)*0.6)
second = int(len(features)*0.8)
train = features.iloc[:first]
validation = features.iloc[first:second]
reserved_test = features.iloc[second:]
print('Rows:', len(train), len(validation), len(reserved_test))
assert train.index.max() < validation.index.min()
assert validation.index.max() < reserved_test.index.min()
display(features.head())

## Avoid leakage and unsupported claims

- Do not use end time or a completed duration to forecast information needed before a ride.
- Observed rentals measure recorded usage, which can be limited by bike availability. They do not measure all unmet demand.
- Preserve New York local-time semantics for weekday/hour features and document DST handling.
- A hypothesis about weekly patterns does not establish causation. Report uncertainty and compare held-out forecast errors.
- Write the final modeling and model comparison notebooks only after agreeing scope and running the EDA.